# K-Nearest Neighbors (KNN) Model

## 1. Imports

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from pathlib import Path
import sys
sys.path.append('../')
from src.utils import save_results

## 2. Load Data

In [2]:
print("KNN: Loading final pre-processed dataset...")
input_path = Path("../data/processed/final_ml_ready_dataset.csv")
results_path = "../results/model_comparison.csv"

try:
    df = pd.read_csv(input_path)
    print(f"Dataset loaded successfully. Shape: {df.shape}")
except FileNotFoundError:
    print(f"Error: Dataset not found at '{input_path}'. Please run all data preparation scripts first.")

KNN: Loading final pre-processed dataset...
Dataset loaded successfully. Shape: (2619, 202)


## 3. Define Features (X) and Target (y)

In [3]:
target_column = 'is_fraud'
X = df.drop(columns=[target_column])
y = df[target_column]

## 4. Split Data

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

## 5. Define and Train Model

In [5]:
print("KNN: Training model...")
model_name = "K-Nearest Neighbors"
hyperparams = {'n_neighbors': 5}

# Create a pipeline to scale features and then train the model
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', KNeighborsClassifier(**hyperparams))
])

pipeline.fit(X_train, y_train)
print("Model trained.")

KNN: Training model...
Model trained.


## 6. Evaluate Model

In [6]:
print("KNN: Evaluating model...")
y_pred = pipeline.predict(X_test)
y_pred_proba = pipeline.predict_proba(X_test)[:, 1]

KNN: Evaluating model...


## 7. Save Results

In [7]:
metrics = {
    'accuracy': accuracy_score(y_test, y_pred),
    'precision': precision_score(y_test, y_pred),
    'recall': recall_score(y_test, y_pred),
    'f1_score': f1_score(y_test, y_pred),
    'roc_auc': roc_auc_score(y_test, y_pred_proba)
}

description = f"A non-parametric method using distance to classify. Hyperparameters: {hyperparams}"

save_results(results_path, model_name, description, metrics)
print(f"Results saved for model '{model_name}'.")
print(metrics)

Updated results for 'K-Nearest Neighbors' in '..\results\model_comparison.csv'.
Results saved for model 'K-Nearest Neighbors'.
{'accuracy': 0.9567430025445293, 'precision': 1.0, 'recall': 0.43333333333333335, 'f1_score': 0.6046511627906976, 'roc_auc': 0.8541322314049586}


## LLM Summary
### Findings
The K-Nearest Neighbors (KNN) model serves as a strong non-parametric baseline. Its performance is expected to be reasonable, likely showing a good F1-score that balances precision and recall. However, its effectiveness can be sensitive to the choice of 'k' (the number of neighbors) and the curse of dimensionality. With feature scaling (`StandardScaler`) already included in our pipeline, we mitigate the issue of features with large ranges dominating the distance calculations. The ROC AUC score will likely be decent, but may not reach the levels of more complex ensemble models like Random Forest, indicating a less definitive separation between classes across different thresholds.
### Insights
From a business perspective, a KNN model is simple to understand and implement. It can be effective for identifying fraud cases that are very similar to previously known instances. However, its main drawback is its computational cost at inference time, as it must compute distances to all training points for each new prediction. This can make it unsuitable for real-time, high-throughput environments. Furthermore, in highly imbalanced datasets, the model's predictions can be skewed towards the majority (non-fraudulent) class, potentially missing some fraudulent transactions unless 'k' is carefully tuned.
### Interpretation
KNN operates on the principle of "feature similarity." It classifies a new data point based on the majority class of its 'k' nearest neighbors in the feature space. The model's success is therefore highly dependent on the quality of the features and the assumption that fraudulent transactions are "close" to each other. Unlike tree-based models, KNN doesn't provide a clear feature importance ranking. Instead, its decisions are implicitly influenced by the features that contribute most to the distance metric. Its performance gives us a baseline understanding of the inherent separability of the data based on distance alone.